In [1]:
%load_ext autoreload
%autoreload 2

Here we make the connection to the database to get the datas we scraped from polimarket.

In [2]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

We read in the event_rows do an events_df which contains the id, title and description of the events.

In [3]:
import pandas as pd
# fetch all rows and convert to DataFrame
event_rows = await conn.fetch('SELECT id, title, description FROM "Event";')
events_df = pd.DataFrame([dict(r) for r in event_rows])
print(f'Loaded {len(events_df)} rows into comment_df')
events_df.set_index('id', inplace=True)
events_df.head()

Loaded 65056 rows into comment_df


,title,description
id,,
3407,Will Coinbase’s NFT marketplace be live by Feb...,This is a market on whether Coinbase’s NFT pla...
5809,Will Bored Apes or CryptoPunks have a higher f...,This is a market group on whether Bored Apes o...
5773,Will the CDC declare a variant of high consequ...,This is a market group on whether the CDC will...
4999,Will Coinbase’s NFT marketplace be live by Apr...,This is a market on whether Coinbase’s NFT pla...
5804,Will Coinbase’s NFT marketplace be live by...?,This is a market group on the when Coinbase’s ...


For the embedding we used the all-MiniLM-L6-v2 embedding model from BERT.

In [4]:
from bertopic import BERTopic
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2")

/Users/dhanna/miniconda3/envs/polymarket/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We get the events on which the users betted on. Each row contains an address (which is the users' ID), the name of the users, an event_id they betted on, the title of that event and the description of that event

In [5]:
import pandas as pd
# fetch all rows and convert to DataFrame
user_event_associations = await conn.fetch('SELECT * FROM user_event_associations;')
user_event_associations_df = pd.DataFrame([dict(r) for r in user_event_associations])
print(f'Loaded {len(user_event_associations_df)} rows into user_event_associations_df_df')
user_event_associations_df.head()

Loaded 483767 rows into user_event_associations_df_df


,address,name,pseudonym,event_id,event_title,event_slug,event_description
0,0x0e241120ea963237d6315b9ace28f4b664f7f81d,Hulkiorra,Attentive-Bank,24423,Bitcoin Up or Down on May 15?,bitcoin-up-or-down-on-may-15,"This market will resolve to ""Up"" if the ""Close..."
1,0x0e241120ea963237d6315b9ace28f4b664f7f81d,Hulkiorra,Attentive-Bank,24663,Bitcoin strikes on May 23?,bitcoin-strikes-on-may-23,"This market will resolve to ""Yes"" if the Binan..."
2,0x0e241120ea963237d6315b9ace28f4b664f7f81d,Hulkiorra,Attentive-Bank,24945,"Bitcoin Price - May 22, 5PM ET",bitcoin-price-may-22-5pm-et,This market will resolve according to the fina...
3,0x0e241120ea963237d6315b9ace28f4b664f7f81d,Hulkiorra,Attentive-Bank,25075,Ethereum Up or Down on May 26?,ethereum-up-or-down-on-may-26,"This market will resolve to ""Up"" if the ""Close..."
4,0x0e241120ea963237d6315b9ace28f4b664f7f81d,Hulkiorra,Attentive-Bank,25113,Solana price on May 30?,solana-price-on-may-30,This market will resolve according to the fina...


As we scraped the events and the events which the users betted on at a different time, they may not contain the same events. We get the events IDs, titles and descriptions from the users too.

In [6]:
# Create events_df from user_event_associations_df
# Extract unique events with their titles and descriptions
events_from_users = user_event_associations_df[['event_id', 'event_title', 'event_description']].drop_duplicates()

# Rename columns to match original events_df structure
events_from_users = events_from_users.rename(columns={
    'event_id': 'id',
    'event_title': 'title',
    'event_description': 'description'
})

# Convert id to int and set as index
events_from_users['id'] = events_from_users['id'].astype(int)
events_from_users = events_from_users.set_index('id')

print(f'Extracted {len(events_from_users)} unique events from user associations')
print(f'Event ID range: {events_from_users.index.min()} to {events_from_users.index.max()}')
print()
events_from_users.head()

Extracted 33394 unique events from user associations
Event ID range: 3854 to 903799



,title,description
id,,
24423,Bitcoin Up or Down on May 15?,"This market will resolve to ""Up"" if the ""Close..."
24663,Bitcoin strikes on May 23?,"This market will resolve to ""Yes"" if the Binan..."
24945,"Bitcoin Price - May 22, 5PM ET",This market will resolve according to the fina...
25075,Ethereum Up or Down on May 26?,"This market will resolve to ""Up"" if the ""Close..."
25113,Solana price on May 30?,This market will resolve according to the fina...


We concatanate the events' title and description and use the BERT model to embed them.

In [8]:
from sentence_transformers import SentenceTransformer

# Load the same embedding model used by topic_model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


print(f"Original events_df: {len(events_df)} events")
print(f"Events from user associations: {len(events_from_users)} events")

# Concatenate the dataframes
# Use concat to combine them, keeping only events that don't already exist in events_df
combined_events = pd.concat([events_df, events_from_users])

# Remove duplicates (keep first occurrence - from original events_df)
combined_events = combined_events[~combined_events.index.duplicated(keep='first')]


# Update events_df to the combined version
events_df = combined_events

# Create combined text for embeddings
events_df['combined_text'] = events_df['title'].fillna('') + " " + events_df['description'].fillna('')

embeddings = embedding_model.encode(events_df['combined_text'].tolist(), show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Number of events with embeddings: {len(embeddings)}")

Original events_df: 65056 events
Events from user associations: 33394 events


Batches: 100%|██████████| 3077/3077 [03:58<00:00, 12.90it/s]



Embeddings shape: (98450, 384)
Number of events with embeddings: 98450


Then we made a semantic similarity search system for events using FAISS. The find_similar_events function returns the top k similar events with their titles and similarity scores. We put the definition of find_similar_events function to recommend_events_for_user_by_topic_based.py

In [10]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
index.add(normalized_embeddings.astype('float32'))

index_to_id = {i: id for i, id in enumerate(events_df.index)}
id_to_index = {id: i for i, id in enumerate(events_df.index)}


Here is an an example where we list the top 5 similar events for an event.

In [12]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import find_similar_events

k = 5
example_id = events_df.index[6000]
print(f"Finding events similar to: {events_df.loc[example_id, 'title']}\n")
k=5
similar_events = find_similar_events(id_to_index, normalized_embeddings,index,index_to_id, events_df, example_id, k)
for i, event in enumerate(similar_events, 1):
    print(f"{i}. {event['title']} (similarity: {event['similarity']:.4f})")

Finding events similar to: World Series: Yankees vs. Dodgers Game 3

1. World Series: Yankees vs. Dodgers Game 3 (similarity: 1.0000)
2. World Series: Dodgers vs. Yankees Game 1 (similarity: 0.9281)
3. World Series: Dodgers vs. Yankees Game 1 (similarity: 0.9281)
4. World Series: Yankees vs. Dodgers Game 4 (similarity: 0.9103)
5. World Series: Yankees vs. Dodgers Game 4 (similarity: 0.9103)


Here we select a user and make recommended events for him/her based on the event he/she betted on in the past. We weight a recommendation for an event more if it is recommended because of multiple events.

In [13]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
from pathlib import Path
# Get a specific user (using the first user as an example)
example_user = user_event_associations_df.iloc[0]['address']
example_name = user_event_associations_df.iloc[0]['name']

print(f"User: {example_name} ({example_user})")
print()

# Get all event_ids for this user
user_events = user_event_associations_df[user_event_associations_df['address'] == example_user]
event_ids = user_events['event_id'].unique()

print(f"Total events betted on: {len(event_ids)}")
print(f"\nEvent IDs: {event_ids}")
print(f"\nSample of events:")
print(user_events[['event_id', 'event_title']].head(10))

top_n=10
similar_per_event=5
print(f"\nRecommended events for this user::")
recommendations= recommend_events_for_user_by_topic_based(id_to_index, normalized_embeddings,index,index_to_id, user_event_associations_df,events_df,example_user , top_n, similar_per_event)
if recommendations:
    for rank, rec in enumerate(recommendations, 1):
        print(f"{rank}. [{rec['id']}] {rec['title']}")


User: Hulkiorra (0x0e241120ea963237d6315b9ace28f4b664f7f81d)

Total events betted on: 61

Event IDs: ['24423' '24663' '24945' '25075' '25113' '25286' '25300' '25432' '28256'
 '30321' '30353' '34325' '50713' '903378' '903691' '11322' '12795' '13551'
 '13813' '14199' '14465' '14526' '14628' '14641' '14657' '14820' '14895'
 '14896' '14906' '14992' '15021' '15025' '15033' '15034' '15169' '15393'
 '15397' '15401' '16219' '16251' '18656' '19583' '19586' '21366' '21781'
 '21927' '22177' '22482' '22518' '22519' '22526' '22632' '22655' '23072'
 '23206' '23691' '23826' '24165' '24181' '24182' '24185']

Sample of events:
  event_id                                        event_title
0    24423                      Bitcoin Up or Down on May 15?
1    24663                         Bitcoin strikes on May 23?
2    24945                     Bitcoin Price - May 22, 5PM ET
3    25075                     Ethereum Up or Down on May 26?
4    25113                            Solana price on May 30?
5    25286

We make a recommendations_df, where we make a recommendation of events for each users.

In [14]:
# Generate recommendations for ALL users
from tqdm.auto import tqdm

# Get all unique users
unique_users = user_event_associations_df[['address', 'name', 'pseudonym']].drop_duplicates()
print(f"Total unique users: {len(unique_users)}")
print()

# Store all recommendations
all_recommendations = []

# Process each user
for idx, user_row in tqdm(unique_users.iterrows(), total=len(unique_users), desc="Generating recommendations"):
    user_address = user_row['address']
    user_name = user_row['name']
    user_pseudonym = user_row['pseudonym']
    
    # Generate recommendations for this user
    recommendations = recommend_events_for_user_by_topic_based(id_to_index, normalized_embeddings,index,index_to_id, user_event_associations_df,events_df, user_address, top_n,  similar_per_event)
    
    # Store results for each recommended event
    for rank, rec in enumerate(recommendations, 1):
        all_recommendations.append({
            'user_address': user_address,
            'user_name': user_name,
            'user_pseudonym': user_pseudonym,
            'rank': rank,
            'recommended_event_id': rec['id'],
            'recommended_event_title': rec['title'],
            'total_score': rec['total_score'],  # Higher = similar to more user events
            'avg_similarity': rec['avg_similarity'],
            'mention_count': rec['count']
        })

# Convert to DataFrame
recommendations_df = pd.DataFrame(all_recommendations)
print(f"\nGenerated {len(recommendations_df)} total recommendations")
print(f"Average recommendations per user: {len(recommendations_df) / len(unique_users):.2f}")
print()
print("Sample recommendations:")
recommendations_df.head(20)

Total unique users: 24640



Generating recommendations:   0%|          | 113/24640 [00:07<27:32, 14.84it/s]


KeyboardInterrupt: 

## Train/Test Split - Hold Out Last 20% of Each User's Bets

In [18]:
# Hold out the last 20% of bets for each user

train_data = []
test_data = []

# Group by user address
grouped = user_event_associations_df.groupby('address')

for user_address, user_bets in grouped:
    # Sort by index to maintain chronological order
    user_bets = user_bets.sort_index()
    
    # Calculate split point (80% train, 20% test)
    n_bets = len(user_bets)
    split_point = int(n_bets * 0.8)
    
    # Split the data
    train_bets = user_bets.iloc[:split_point]
    test_bets = user_bets.iloc[split_point:]
    
    train_data.append(train_bets)
    test_data.append(test_bets)

# Concatenate all splits
train_user_events = pd.concat(train_data, ignore_index=True)
test_user_events = pd.concat(test_data, ignore_index=True)


Baseline Recommenders: We create simple baseline recommenders to compare against the topic-based recommender, we recommend the most popular bets on Polimarket.

In [22]:
# Calculate event popularity based on training data
# Popularity = number of unique users who bet on each event

event_popularity = train_user_events.groupby('event_id')['address'].nunique().reset_index()
event_popularity.columns = ['event_id', 'num_users']
event_popularity = event_popularity.sort_values('num_users', ascending=False)

print(f"Top 20 most popular events (based on training data):")

# Merge with event details to show titles
popular_events_with_titles = event_popularity.head(20).merge(
    train_user_events[['event_id', 'event_title']].drop_duplicates(),
    on='event_id',
    how='left'
)

for idx, row in popular_events_with_titles.iterrows():
    print(f"{row['num_users']:>4} users : [{row['event_id']}] {row['event_title']}")
    
print()
print(f"Total unique events in training data: {len(event_popularity)}")

Top 20 most popular events (based on training data):
1759 users : [16108] Russia x Ukraine ceasefire in 2025?
1547 users : [11439] Super Bowl Champion 2025
1518 users : [16100] Highest grossing movie in 2025?
1425 users : [16096] What price will Bitcoin hit in 2025?
1152 users : [16092] US recession in 2025?
1138 users : [11322] Will Biden finish his term?
1009 users : [16180] Maduro out by...?
 998 users : [10223] Balance of Power: 2024 Election
 996 users : [16105] Khamenei out as Supreme Leader of Iran in 2025?
 986 users : [23656] Super Bowl Champion 2026
 955 users : [10656] Who wins Presidency + Popular Vote?
 953 users : [11696] Fed Interest Rates: September 2024
 920 users : [19696] F1 Drivers Champion
 911 users : [11385] Democratic VP nominee?
 891 users : [16097] What price will Ethereum hit in 2025?
 876 users : [23947] Chile Presidential Election
 858 users : [14023] Who will be inaugurated as President? 
 851 users : [30829] Democratic Presidential Nominee 2028
 850 users

In [27]:
def recommend_popular_events(user_address, train_data, event_popularity, top_n=10):
    # Get events this user has already bet on (from training data)
    user_events = train_data[train_data['address'] == user_address]['event_id'].unique()
    user_events_set = set(user_events)
    
    # Filter out events the user has already bet on
    available_events = event_popularity[~event_popularity['event_id'].isin(user_events_set)]
    
    # Return top N most popular events
    recommendations = available_events.head(top_n)['event_id'].tolist()
    
    return recommendations

# Test the popularity-based recommender on the example user
popular_recs = recommend_popular_events(example_user, train_user_events, event_popularity, top_n=10)

print(f"Popularity-based recommendations for user: {example_name} ({example_user})")

# Get event titles for display
for idx, event_id in enumerate(popular_recs, 1):
    event_title = train_user_events[train_user_events['event_id'] == event_id]['event_title'].iloc[0]
    num_users = event_popularity[event_popularity['event_id'] == event_id]['num_users'].iloc[0]
    print(f"{idx:2}. [{event_id}] {event_title}")
    print(f"    (Bet on by {num_users} users)")
    print()

Popularity-based recommendations for user: endo2 (0xf3df46b4a1dca573797c034cd4d1dc60fa660324)
 1. [16108] Russia x Ukraine ceasefire in 2025?
    (Bet on by 1759 users)

 2. [11439] Super Bowl Champion 2025
    (Bet on by 1547 users)

 3. [16100] Highest grossing movie in 2025?
    (Bet on by 1518 users)

 4. [16096] What price will Bitcoin hit in 2025?
    (Bet on by 1425 users)

 5. [16092] US recession in 2025?
    (Bet on by 1152 users)

 6. [11322] Will Biden finish his term?
    (Bet on by 1138 users)

 7. [16180] Maduro out by...?
    (Bet on by 1009 users)

 8. [10223] Balance of Power: 2024 Election
    (Bet on by 998 users)

 9. [16105] Khamenei out as Supreme Leader of Iran in 2025?
    (Bet on by 996 users)

10. [23656] Super Bowl Champion 2026
    (Bet on by 986 users)



In [29]:
# Generate topic-based recommendations using ONLY training data
print(f"Topic-based recommendations for user: {example_name} ({example_user})")

# Use train_user_events instead of user_event_associations_df
topic_recommendations = recommend_events_for_user_by_topic_based( id_to_index,  normalized_embeddings, index, index_to_id, train_user_events, events_df, example_user, top_n=20, similar_per_event=5)

if topic_recommendations:
    for rank, rec in enumerate(topic_recommendations, 1):
        print(f"{rank:2}. [{rec['id']}] {rec['title']}")


Topic-based recommendations for user: endo2 (0xf3df46b4a1dca573797c034cd4d1dc60fa660324)
 1. [57723] 2nd Largest company end of 2025?
 2. [45221] Trump strikes another drug boat by Sep 30?
 3. [41835] When will Israel raid Gaza aid flotilla?
 4. [37155] Will Russia capture Myrnohrad by August 31?
 5. [51135] Will Russia capture Stepanivka by October 31?
 6. [12869] NYC mayoral special election in 2024?
 7. [34022] Will Trump mention "South Park" by Sunday?
 8. [45222] Israel next strike on Yemen on...?
 9. [42626] Houthi strike on Israel by...?
10. [63758] U.S. anti-cartel operation on foreign soil by December 31?
11. [49825] Israel next strike on Yemen on...?
12. [36708] U.S. anti-cartel operation on foreign soil by September 30?
13. [56968] Will Hamas release any more hostages by October 12?
14. [61883] Israel strikes Gaza by...?
15. [52186] Israel strikes Iran by October 3?
16. [22978] Will Russia capture Siversk by August 31?
17. [56710] Will Hamas release any more hostages by Octo

Check which recommended events the user actually bet on in the test set (held-out data).

In [30]:
# Get the events the user actually bet on in the test set
test_user_bets = test_user_events[test_user_events['address'] == example_user]
actual_test_events = set(test_user_bets['event_id'].unique())

print(f"User {example_name} bet on {len(actual_test_events)} events in the test set")

# Check topic-based recommendations
topic_rec_ids = [str(rec['id']) for rec in topic_recommendations]
topic_hits = set(topic_rec_ids) & actual_test_events

print("Topic-Based Recommender:")
print(f"  Recommendations: {len(topic_rec_ids)}")
print(f"  Hits (in test set): {len(topic_hits)}")
print(f"  Hit rate: {len(topic_hits)/len(topic_rec_ids)*100:.1f}%")
if topic_hits:
    print(f"  Events that matched: {topic_hits}")
print()

# Check popularity-based recommendations
popularity_hits = set(popular_recs) & actual_test_events

print("Popularity-Based Baseline:")
print(f"  Recommendations: {len(popular_recs)}")
print(f"  Hits (in test set): {len(popularity_hits)}")
print(f"  Hit rate: {len(popularity_hits)/len(popular_recs)*100:.1f}%")
if popularity_hits:
    print(f"  Events that matched: {popularity_hits}")
print()

print("Comparison:")
print(f"  Topic-based is {'better' if len(topic_hits) > len(popularity_hits) else 'worse' if len(topic_hits) < len(popularity_hits) else 'equal'} than popularity baseline for this user")

User endo2 bet on 19 events in the test set
Topic-Based Recommender:
  Recommendations: 20
  Hits (in test set): 1
  Hit rate: 5.0%
  Events that matched: {'59799'}

Popularity-Based Baseline:
  Recommendations: 10
  Hits (in test set): 0
  Hit rate: 0.0%

Comparison:
  Topic-based is better than popularity baseline for this user


We make recommendations to all the users by their first 80% of their bets. After that we compare it to the baseline recommendations and the 20% of their bets which we witheld.

In [32]:
'''
from tqdm.auto import tqdm

# Get unique users who have events in both train and test sets
train_users = set(train_user_events['address'].unique())
test_users = set(test_user_events['address'].unique())
users_to_evaluate = train_users & test_users

print(f"Users with events in both train and test sets: {len(users_to_evaluate)}")
print()

# Store results for each user
results = []

# Evaluate for all users
for user_address in tqdm(list(users_to_evaluate)):
    # Get test events for this user
    user_test_events = set(test_user_events[test_user_events['address'] == user_address]['event_id'].unique())
    
    # Skip users with no test events
    if len(user_test_events) == 0:
        continue
    
    # Generate topic-based recommendations
    topic_recs = recommend_events_for_user_by_topic_based( id_to_index,  normalized_embeddings, index, index_to_id, train_user_events,events_df, user_address, top_n=10, similar_per_event=5)
    topic_rec_ids = set([str(rec['id']) for rec in topic_recs])
    topic_hits = len(topic_rec_ids & user_test_events)
    topic_precision = topic_hits / len(topic_rec_ids) if len(topic_rec_ids) > 0 else 0

    
    # Generate popularity-based recommendations
    pop_recs = recommend_popular_events(user_address, train_user_events, event_popularity, top_n=10)
    pop_rec_ids = set(pop_recs)
    pop_hits = len(pop_rec_ids & user_test_events)
    pop_precision = pop_hits / len(pop_rec_ids) if len(pop_rec_ids) > 0 else 0
    
    # Store results
    results.append({
        'user_address': user_address,
        'num_test_events': len(user_test_events),
        'topic_hits': topic_hits,
        'topic_precision': topic_precision,
        'pop_hits': pop_hits,
        'pop_precision': pop_precision
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.head()
'''

Users with events in both train and test sets: 19006



100%|██████████| 19006/19006 [55:04<00:00,  5.75it/s]   


,user_address,num_test_events,topic_hits,topic_precision,pop_hits,pop_precision
0,0x4c5f9c091182c88f1414608a36d0cfbd8ccfaa90,1,0,0.0,0,0.0
1,0xea2c936694d56cf55454d32149ae9c5a45b7f0f6,1,0,0.0,0,0.0
2,0x6ccc115a898c67274d9b70252dd94b83859737a7,20,0,0.0,0,0.0
3,0x9d76fce36508261e0aa89c3530128a67823f5586,2,0,0.0,0,0.0
4,0x0dd250f0b25d0981c42cd43662e7043c3a5d06df,46,0,0.0,0,0.0


We then saves the results into files.

In [37]:
'''
import json

with open('topic_based_recommender_test_results.json', 'w') as f:
    json.dump(results, f)

results_df.to_csv('topic_based_recommender_test_results_df.csv', index=False)
'''

In [15]:
import json
results_df = pd.read_csv('topic_based_recommender_test_results_df.csv')
with open('topic_based_recommender_test_results.json', 'r') as f:
    results = json.load(f)

In [16]:
results_df

,user_address,num_test_events,topic_hits,topic_precision,pop_hits,pop_precision
0,0x4c5f9c091182c88f1414608a36d0cfbd8ccfaa90,1,0,0.0,0,0.0
1,0xea2c936694d56cf55454d32149ae9c5a45b7f0f6,1,0,0.0,0,0.0
2,0x6ccc115a898c67274d9b70252dd94b83859737a7,20,0,0.0,0,0.0
3,0x9d76fce36508261e0aa89c3530128a67823f5586,2,0,0.0,0,0.0
4,0x0dd250f0b25d0981c42cd43662e7043c3a5d06df,46,0,0.0,0,0.0
...,...,...,...,...,...,...
19001,0x0d5040535023424d1928c5b850d63de5dd736a53,3,0,0.0,0,0.0
19002,0x94e6d385757c68b4f9f239b2d5ea8216f2e40947,24,0,0.0,0,0.0
19003,0xf1837033ddfe041c624e2414dbe68fa961b33be7,2,0,0.0,0,0.0
19004,0x4031382e6b1821df15ef1bb500e4cd58fb338343,1,0,0.0,0,0.0


In [17]:
# Average metrics
print("\nAverage Precision@10:")
print(f"  Topic-Based:  {results_df['topic_precision'].mean():.4f}")
print(f"  Popularity:   {results_df['pop_precision'].mean():.4f}")

# Total hits
print("\nTotal Hits:")
print(f"  Topic-Based:  {results_df['topic_hits'].sum()} hits")
print(f"  Popularity:   {results_df['pop_hits'].sum()} hits")


# Average hits per user
print("Average Hits per User:")
print(f"  Topic-Based:  {results_df['topic_hits'].mean():.2f}")
print(f"  Popularity:   {results_df['pop_hits'].mean():.2f}")
print()


if results_df['topic_precision'].mean() > results_df['pop_precision'].mean():
    improvement = ((results_df['topic_precision'].mean() / results_df['pop_precision'].mean()) - 1) * 100
    print(f"Topic-based recommender outperforms popularity baseline by {improvement:.1f}%")
elif results_df['topic_precision'].mean() < results_df['pop_precision'].mean():
    decline = ((results_df['pop_precision'].mean() / results_df['topic_precision'].mean()) - 1) * 100
    print(f"Topic-based recommender underperforms popularity baseline by {decline:.1f}%")
else:
    print("Both recommenders perform equally")


Average Precision@10:
  Topic-Based:  0.0151
  Popularity:   0.0012

Total Hits:
  Topic-Based:  2152 hits
  Popularity:   230 hits
Average Hits per User:
  Topic-Based:  0.11
  Popularity:   0.01

Topic-based recommender outperforms popularity baseline by 1150.7%
